# E10-2 — Which audio features help an SVM recognise instruments?

Reuse E9’s Freesound collection code to build a balanced ten-instrument dataset. Compare representations by how accurately the same supervised classifier predicts the instrument of a new recording.

Follow the feature comparison in the [Freesound SVM core example](../examples/freesound-classification-core.ipynb) and the listening and descriptor plots in its [visualization companion](../examples/freesound-classification-visualization.ipynb). You will need scikit-learn for classification and Essentia for local feature extraction. To reuse E9 downloads, copy curated class folders into the separate E10 collection directory.

In [ ]:
from collections import Counter
from getpass import getpass
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import requests
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from urllib3.util.retry import Retry

# Use the current API names and feature selection from the classification example.
SCALAR_DESCRIPTORS = [
    'spectral_centroid', 'spectral_energy', 'dissonance',
    'pitch_salience', 'inharmonicity', 'log_attack_time',
]
feature_names = SCALAR_DESCRIPTORS + [f'mfcc.{i}' for i in range(13)]
descriptorMapping = dict(enumerate(feature_names))
API_DESCRIPTORS = SCALAR_DESCRIPTORS + ['mfcc']

targetDir = Path('testDownload-E10') / 'collections'
queryDir = Path('testDownload-E10') / 'queries'


def freesound_session():
    load_dotenv()
    api_key = (os.getenv('FREESOUND_API_KEY') or getpass('Freesound API key: ')).strip()
    if not api_key:
        raise ValueError('Enter a Freesound API key to download sounds.')
    return make_session(api_key)

In [ ]:
BASE_URL = 'https://freesound.org/apiv2'


def make_session(api_key):
    session = requests.Session()
    session.headers.update({'Authorization': f'Token {api_key}'})
    # Retry occasional server/rate-limit failures, respecting Retry-After.
    retries = Retry(total=3, backoff_factor=0.5,
                    status_forcelist=[429, 500, 502, 503, 504],
                    allowed_methods=['GET'], respect_retry_after_header=True)
    session.mount('https://', HTTPAdapter(max_retries=retries))
    return session


def api_get_json(session, endpoint, *, params=None):
    url = endpoint if endpoint.startswith('https://freesound.org/apiv2/') else BASE_URL + endpoint
    response = session.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()


def feature_vector(analysis, names):
    """Convert current APIv2 descriptor JSON to a selected 1D numerical vector."""
    values = []
    for name in names:
        if '.' in name:  # For example, mfcc.3
            descriptor, index = name.rsplit('.', 1)
            vector = analysis.get(descriptor)
            if not isinstance(vector, (list, tuple)) or len(vector) <= int(index):
                raise ValueError(f'Missing/short descriptor: {descriptor}')
            value = vector[int(index)]
        else:
            value = analysis.get(name)
        if value is None or isinstance(value, (list, tuple, dict, bool)):
            raise ValueError(f'Missing or non-scalar descriptor: {name}')
        value = float(value)
        if not np.isfinite(value):
            raise ValueError(f'Non-finite descriptor: {name}')
        values.append(value)
    return np.asarray(values, dtype=float)


def download_preview(preview_url, target):
    """Previews may be hosted externally: do NOT send the API token to that URL."""
    if not preview_url:
        return
    with requests.get(preview_url, stream=True, timeout=30) as response:
        response.raise_for_status()
        with target.open('wb') as handle:
            for chunk in response.iter_content(chunk_size=64 * 1024):
                if chunk:
                    handle.write(chunk)


def download_class_sounds(session, query, tag, count, target_root, duration,
                          save_previews=True, max_candidates=200, cache_folder=None):
    """Cache at most `count` valid sounds per class, without deleting existing files."""
    class_dir = target_root / (cache_folder or query)
    class_dir.mkdir(parents=True, exist_ok=True)
    existing_ids = set()
    for path in class_dir.glob('*/features.json'):
        try:
            feature_vector(json.loads(path.read_text(encoding='utf-8')), feature_names)
        except (ValueError, TypeError):
            continue
        existing_ids.add(path.parent.name)
    if len(existing_ids) >= count:
        print(f'{query}: {len(existing_ids)} cached sounds; no download needed.')
        return

    filters = [f'duration:[{duration[0]} TO {duration[1]}]']
    if tag:
        filters.insert(0, f'tag:{tag}')

    page = 1
    seen = 0
    added = 0
    skipped = 0
    while len(existing_ids) < count and seen < max_candidates:
        payload = api_get_json(session, '/search/', params={
            'query': query,
            'filter': ' '.join(filters),
            'sort': 'score',
            'fields': 'id,name,url,username,license,previews',
            'page_size': 30,
            'page': page,
        })
        results = payload.get('results', [])
        if not results:
            break
        for sound in results:
            if len(existing_ids) >= count or seen >= max_candidates:
                break
            seen += 1
            sound_id = str(sound['id'])
            if sound_id in existing_ids:
                continue
            sound_dir = class_dir / sound_id
            feature_file = sound_dir / 'features.json'
            try:
                analysis = api_get_json(session, f'/sounds/{sound_id}/analysis/',
                                        params={'fields': ','.join(API_DESCRIPTORS)})
                # Require the lab's descriptor fields so all experiments use the same sounds.
                feature_vector(analysis, feature_names)
            except requests.HTTPError as exc:
                if exc.response.status_code != 404:
                    raise
                skipped += 1
                if skipped <= 3:
                    print(f'  No analysis for {sound_id}.')
                continue
            except (ValueError, KeyError, TypeError) as exc:
                skipped += 1
                if skipped <= 3:
                    print(f'  Skipping {sound_id}: {type(exc).__name__}: {exc}')
                continue

            sound_dir.mkdir(parents=True, exist_ok=True)
            with feature_file.open('w', encoding='utf-8') as f:
                json.dump(analysis, f, indent=2)
            with (sound_dir / 'metadata.json').open('w', encoding='utf-8') as f:
                json.dump({k: sound.get(k) for k in
                           ['id', 'name', 'url', 'username', 'license']}, f, indent=2)
            if save_previews:
                preview = (sound.get('previews') or {}).get('preview-hq-mp3')
                try:
                    download_preview(preview, sound_dir / 'preview.mp3')
                except requests.RequestException as exc:
                    (sound_dir / 'preview.mp3').unlink(missing_ok=True)
                    print(f'  Preview unavailable for {sound_id}: {type(exc).__name__}')
            existing_ids.add(sound_id)
            added += 1
        if not payload.get('next'):
            break
        page += 1
    print(f'{query}: {len(existing_ids) - added} cached + {added} new; {skipped} incomplete/failed.')


def load_dataset(root, selected_names, class_filter=None):
    features, labels, ids, paths = [], [], [], []
    seen_ids = set()
    # Only files named features.json from this API version.
    for file_path in sorted(Path(root).glob('*/*/features.json')):
        if class_filter and file_path.parent.parent.name != class_filter:
            continue
        if file_path.parent.name in seen_ids:
            print(f'Skipping duplicate Freesound ID {file_path.parent.name} to prevent data leakage.')
            continue
        try:
            analysis = json.loads(file_path.read_text(encoding='utf-8'))
            vector = feature_vector(analysis, selected_names)
        except (ValueError, TypeError, KeyError, json.JSONDecodeError) as exc:
            print(f'Skipping {file_path}: {exc}')
            continue
        features.append(vector)
        labels.append(file_path.parent.parent.name)
        ids.append(file_path.parent.name)
        paths.append(file_path)
        seen_ids.add(file_path.parent.name)
    if not features:
        return np.empty((0, len(selected_names))), np.array([]), [], []
    return np.vstack(features), np.asarray(labels), ids, paths

## 1. Curate the collection

Choose ten instruments and collect 20 single notes or strokes per class. Adapt the example for each instrument. Listen to samples, record the search settings and check the counts after duplicate removal. For cached sounds missing a preview, supply the matching preview or replace the recording before comparing methods.

In [ ]:
# Example collection call. Adapt the query and tag before downloading.
# with freesound_session() as session:
#     download_class_sounds(session, 'violin', 'single-note', 20, targetDir, (0.2, 8))

matrix, labels, ids, paths = load_dataset(targetDir, feature_names)
print(Counter(labels))
audio_paths = [path.with_name('preview.mp3') for path in paths]
if len(set(labels)) != 10 or any(n != 20 for n in Counter(labels).values()):
    raise ValueError('Curate 20 unique recordings for each of ten instruments.')
if not all(path.is_file() for path in audio_paths):
    raise ValueError('Check the preview.mp3 files before continuing.')

## 2. Compare Freesound representations on the training set

Predict the instrument label from the descriptors. Reserve four of each instrument’s twenty recordings for a final test. On the remaining sixteen per class, compare scalar descriptors, MFCCs, and their combination using identical stratified folds. Keep the RBF kernel and its settings fixed throughout the lab. Fit the scaler inside each training fold.

Before running the comparison, name two instruments you expect to confuse and explain which descriptor family might separate them. What accuracy would uniform random guessing achieve here?

In [ ]:
train, test = train_test_split(np.arange(len(labels)), test_size=0.20,
                               stratify=labels, random_state=42)
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
             .split(train, labels[train]))
model = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=10, gamma='scale'))
n_scalar = len(SCALAR_DESCRIPTORS)
representations = {
    'Freesound scalars': matrix[:, :n_scalar],
    'Freesound MFCCs': matrix[:, n_scalar:],
    'Freesound combined': matrix,
}
scores = {}

In [ ]:
def compare_features(name, features):
    values = cross_val_score(model, features[train], labels[train],
                             cv=folds, scoring='accuracy', error_score='raise')
    scores[name] = values
    print(f'{name} ({features.shape[1]} features): {values.mean():.1%} ± {values.std():.1%}')

for name, features in representations.items():
    compare_features(name, features)

In [ ]:
# Inspect two descriptors using training recordings only.
for instrument in np.unique(labels):
    rows = train[labels[train] == instrument]
    plt.scatter(matrix[rows, 0], matrix[rows, 4], label=instrument, alpha=0.7)
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[4])
plt.legend(bbox_to_anchor=(1, 1))
plt.show()

## 3. Test local features and silence removal

Implement the extractor below with Essentia. Decode each preview at a common sample rate, then frame and window the signal. Choose at least two descriptor families (for example, MFCCs and spectral centroid). Compute frame energy before windowing. Plot energy for two training recordings, a quiet note and a note with a sharp attack to choose a threshold relative to each recording’s peak energy.

Return two finite vectors in the same descriptor order. Aggregate all frames for the first and only active frames for the second. Document the units, threshold and aggregation statistics. Flag recordings with no active frames and decide how to handle them using training recordings only. Freeze the extraction settings before processing the test recordings.

In [ ]:
import essentia.standard as es

def local_features(audio_path):
    audio = es.MonoLoader(filename=str(audio_path), sampleRate=44100)()
    # Compute frame energies and descriptors, then aggregate both frame selections.
    raise NotImplementedError('Return (all_frame_features, active_frame_features).')

In [ ]:
pairs = [local_features(path) for path in audio_paths]
for name, vectors in zip(('Essentia all frames', 'Essentia active frames'), zip(*pairs)):
    features = np.vstack(vectors)
    if not np.isfinite(features).all():
        raise ValueError(f'Inspect non-finite values in {name}.')
    representations[name] = features
    compare_features(name, features)

## 4. Evaluate on held-out recordings

Choose the representation with the highest mean validation accuracy, breaking ties by preferring fewer features. Fit it on the full training set and predict the reserved recordings once. Use the confusion matrix to find a recording worth listening to.

In [ ]:
best = max(scores, key=lambda name: (scores[name].mean(), -representations[name].shape[1]))
X = representations[best]
model.fit(X[train], labels[train])
predicted = model.predict(X[test])
print(f'{best}: test accuracy {accuracy_score(labels[test], predicted):.1%}')
print(classification_report(labels[test], predicted, zero_division=0))
ConfusionMatrixDisplay.from_predictions(labels[test], predicted, xticks_rotation=90)
plt.tight_layout()
plt.show()
for row, guess in zip(test, predicted):
    if guess != labels[row]:
        print(ids[row], labels[row], 'predicted as', guess, audio_paths[row])

## 5. Interpret the feature analysis

1. Tabulate feature count and mean ± standard deviation of validation accuracy for all five representations. Did combining scalars and MFCCs help consistently across the paired folds?
2. Compare the two Essentia conditions fold by fold. How did removing quiet frames affect classification? Use one training recording’s energy curve and descriptor values to explain a possible cause.
3. Report the selected representation, its held-out accuracy and the number of correct predictions out of 40. Which instrument had the lowest recall? Use counts from its confusion-matrix row to explain the result.
4. Listen to a misclassified test recording and a training example of the predicted instrument. Give their Freesound IDs and identify an audible similarity. If all predictions were correct, choose a correctly classified pair whose classes overlap in the training scatter plot. What can two plotted descriptors tell you about a decision made in the full feature space?
5. Each instrument contributes only four test recordings. How much would one additional error change its recall? Identify one property of the collection, such as repeated performers or recording sessions, that could affect performance on a new collection.